In [1]:
import numpy as np
import pandas as pd

import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc

import itertools
from scipy.stats import ttest_ind

import sklearn.model_selection
import sklearn.linear_model
import scipy.stats

CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [2]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [3]:
def load_subject_concept(subject_index=2):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/coherence_results"
    file_path = os.path.join(dir, f"coherence_results_{subject_index}.csv")
    df = pd.read_csv(file_path)
    # Load connectivity data for the specified subject

    
    
    # Filter out trials that should be ignored (first 150 trials)
    filtered_data = df[df['trial'] >= 150]
    #filtered_data = filtered_data.drop(columns=['subject_index', 'ch_index1', 'ch_index2'])
    
    # Extract relevant information: trial index, channel combinations, and concept values

    return filtered_data

In [4]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2):
    data_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/explanation_data"
    file_path = os.path.join(data_dir, f"subject_{subject_index}_results.pkl")

    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f)   
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [5]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency"


In [6]:
#df = load_subject_concept(subject_index=2, freq_band='alpha')


In [7]:
#df[(df["frequency_band"] == "alpha") & (df["ch_name1"] == "F3") & (df["ch_name2"] == "C4")]

In [8]:
def load_subject_data(subject_index):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/coherence_results"
    file_path = os.path.join(dir, f"coherence_results_{subject_index}.csv")
    df = pd.read_csv(file_path)


In [14]:
from scipy.stats import pearsonr
def plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept='coherence', band_name="alpha", show_plots=False):
    """
    Plot correlations between predictions and a given concept for ROI channel pairs.
    Only shows plots with absolute correlation > 0.2 and displays each plot individually.
    
    Args:
        df: DataFrame containing the connectivity data
        predictions: Array of prediction values
        roi_channel_pairs: List of channel pairs to analyze
        n_cols: Number of columns in the subplot grid (default=4)
    """
    # Filter dataframe to only include ROI channel pairs
    # Filter by frequency band first (this is often faster)
    band_df = df[df['frequency_band'] == band_name]
 
    
    # Create a DataFrame with all valid channel pairs (including both directions)
    
    #for ch1, ch2 in roi_channel_pairs:
    #    pairs.append((ch1, ch2))
    #    pairs.append((ch2, ch1))  # Add reversed pairs too
    #valid_pairs = pd.DataFrame(pairs, columns=['ch_name1', 'ch_name2'])
    
    # Use merge to efficiently filter for matching channel pairs
    #filtered_df = band_df.merge(valid_pairs, on=['ch_name1', 'ch_name2'], how='inner')

    # Prepare data for correlation analysis
    #trial_indices = sorted(filtered_df['trial'].unique())
    #prediction_values = [predictions[i-151] for i in trial_indices]
    results = {}
    # For each ROI channel pair, create an individual plot if correlation is significant
    for ch1, ch2 in roi_channel_pairs:
        pairs = []
        pairs.append((ch1, ch2))
        pairs.append((ch2, ch1))  # Add reversed pairs too
        valid_pairs = pd.DataFrame(pairs, columns=['ch_name1', 'ch_name2'])
    
    # Use merge to efficiently filter for matching channel pairs
        filtered_df = band_df.merge(valid_pairs, on=['ch_name1', 'ch_name2'], how='inner')
        # Filter data for this channel pair (check both directions)
        pair_data = filtered_df[((filtered_df['ch_name1'] == ch1) & (filtered_df['ch_name2'] == ch2)) | 
                              ((filtered_df['ch_name1'] == ch2) & (filtered_df['ch_name2'] == ch1))]
        

        #pair_trials = pair_data['trial_index'].values
        concept_values = pair_data[concept].values
            
        # Create trial_index to concept value mappin
        if len(concept_values) == 0:
            continue        
        #corr = np.corrcoef(np.array(predictions), np.array(concept_values))[0, 1]
        corr_object = pearsonr(np.array(concept_values), predictions)
        #all_results[(ch1, ch2)] = corr_object
        corr = corr_object[0]
        pval = corr_object[1]
        if pval< (0.05/60):
            results[(ch1, ch2)] = {"corr": corr, "pval": pval}

        
        if show_plots: # Only show plots with absolute correlation > 0.2
            if abs(corr) > 0.2:
                plt.figure(figsize=(6, 4))
                
            # Create scatter plot
                plt.scatter(np.array(concept_values), np.array(predictions), alpha=0.6)
                plt.title(f'{ch1}-{ch2} Correlation with {concept.upper()} (r={corr:.3f}) {band_name}')
                plt.ylabel('Predicted Amplitude')
                plt.xlabel(concept.upper())
                #results[(ch1, ch2)] = corr

    return results

                
    


In [ ]:
#roi_channels = ['C1', 'C3', 'C4', 'C5',  'FC1', 'FC3', 'FC4', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'CP1', 'CP5', 'CP3', 'CP4' ]
#roi_channel_pairs = list(itertools.combinations(roi_channels, 2))
#roi_channels = ch_names
roi_channel_pairs = list(itertools.combinations(roi_channels, 2))



In [12]:
predictions, uncertainties, explanations, ch_names = load_predicted_amplitude_for_subject(subject_index=2)

Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)


In [10]:
plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept='coherence')

NameError: name 'df' is not defined

# coherence all subjects

In [12]:
mne.set_log_level("ERROR")

In [15]:
cfg = load_config()
results_all_subjects = {}
for subject_index in tqdm(cfg.dataset.test_subject_indices):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    roi_channel_pairs = list(itertools.combinations(ch_names, 2))
    results_all_subjects[subject_index] = {}
    print(subject_index)
    df = load_subject_concept(subject_index=subject_index)
    predictions, _, _, _= load_predicted_amplitude_for_subject(subject_index=subject_index)
    for freq_band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        results_all_subjects[subject_index][freq_band] = plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept='coherence', band_name=freq_band)
np.save("results_coherence_all_subjects.npy", results_all_subjects)


  0%|          | 0/32 [00:00<?, ?it/s]

Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
1
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)


  3%|▎         | 1/32 [09:15<4:47:11, 555.86s/it]

Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
2
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)


  6%|▋         | 2/32 [27:31<7:16:39, 873.31s/it]

...

 94%|█████████▍| 30/32 [6:31:58<25:33, 766.51s/it]

Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)
92
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)


 97%|█████████▋| 31/32 [6:41:59<11:56, 716.82s/it]

Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)
102
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)


100%|██████████| 32/32 [6:57:14<00:00, 782.33s/it]


In [11]:
cfg = load_config()
for subject_index in tqdm(cfg.dataset.test_subject_indices):
    print(subject_index)
    df = load_subject_concept(subject_index=subject_index)
    for freq_band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        print(freq_band)
        predictions, _, _, _= load_predicted_amplitude_for_subject(subject_index=subject_index)
        plot_channel_pair_correlations(df, predictions, roi_channel_pairs, concept='lagged_coherence', band_name=freq_band)


  0%|          | 0/32 [00:00<?, ?it/s]

1
delta
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
theta
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
alpha
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
beta
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)
gamma
Loading EEG data...

subject index: 1, frequency band: None

(100, 60, 900)
(1, 60, 1)


  3%|▎         | 1/32 [00:43<22:29, 43.54s/it]

2
delta
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
theta
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
alpha
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
beta
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)
gamma
Loading EEG data...

subject index: 2, frequency band: None

(100, 60, 900)
(1, 60, 1)


  6%|▋         | 2/32 [01:55<30:04, 60.14s/it]

...

 94%|█████████▍| 30/32 [33:20<02:16, 68.17s/it]

92
delta
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)
theta
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)
alpha
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)
beta
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)
gamma
Loading EEG data...

subject index: 92, frequency band: None

(100, 60, 900)
(1, 60, 1)


 97%|█████████▋| 31/32 [34:09<01:02, 62.48s/it]

102
delta
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)
theta
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)
alpha
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)
beta
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)
gamma
Loading EEG data...

subject index: 102, frequency band: None

(100, 60, 900)
(1, 60, 1)


100%|██████████| 32/32 [35:37<00:00, 66.81s/it]


# lagged coherence all subjects